In [1]:
import pandas as pd
import json
import os
import sys
from tqdm import tqdm
import numpy as np

sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())

from scripts.config import (
    COHERE_API_KEY,
    DATA_DIR,
    EMBEDDINGS_1887_COHERE,
    NEWSPAPER_CONFIGS,
    DEBATS_COLLECTION_NAME,
    COHERE_EMBEDDING_MODEL,
)
from scripts.retrieval_utils import (
    RAGRetriever,
    RAGRetriever_force_multicollection,
    find_gold_ranks,
    compute_recall_metrics_dataframe,
    compute_accuracy_at_k,
    compute_mrr_at_k,
    get_existing_ids,
    evaluate_and_print_metrics,
    perform_retrieval,
)
from scripts.utils.chroma_utils import load_chroma_collection
from scripts.utils.json_reading import read_jsonl_basic
from scripts.utils.questions import create_questions_dataset

# Paths
QUESTIONS_DIR = os.path.join(DATA_DIR, "questions")  # NOTE: 'questions', not 'questions_singlehop'
OUTPUT_DIR = os.path.join(DATA_DIR, "retrieval")
EMBEDDINGS_PATH = EMBEDDINGS_1887_COHERE

# Question datasets configuration
QUESTION_DATASETS = [
    {
        'name': 'Generic_LeGaulois_Debats',
        'file': 'MH_Generic_legaulois_v1.jsonl',
        'column_mapping': {
            'id_1': 'journal_id',
            'id_2': 'debat_id',
            'doc_1': 'journal_doc',
            'doc_2': 'debat_doc',
            'metadata_1': 'journal_metadata',
            'metadata_2': 'debat_metadata'
        },
        'source1_name': 'Le Gaulois',
        'source2_name': 'Les Débats',
        'question_type': 'MH - Generic'
    },
    {
        'name': 'Generic_LIntransigeant_Debats',
        'file': 'generated_questions_lintransigeant_all.jsonl',
        'column_mapping': {
            'id_1': 'journal_id',
            'id_2': 'debat_id',
            'doc_1': 'journal_doc',
            'doc_2': 'debat_doc',
            'metadata_1': 'journal_metadata',
            'metadata_2': 'debat_metadata'
        },
        'source1_name': "L'Intransigeant",
        'source2_name': "Les Débats",
        'question_type': 'MH - Generic'
    },
    {
        'name': 'BridgeEntity_LIntransigeant_Debats',
        'file': 'MH_BridgeEntity_lintransigeant_v2.jsonl',
        'column_mapping': {
            'id_1': 'journal_id',
            'id_2': 'debat_id',
            'doc_1': 'journal_doc',
            'doc_2': 'debat_doc',
            'metadata_1': 'journal_metadata',
            'metadata_2': 'debat_metadata'
        },
        'source1_name': "L'Intransigeant",
        'source2_name': 'Les Débats',
        'question_type': 'MH - BridgeEntity'
    },
    {
        'name': 'Comparative_LeGaulois_LIntransigeant',
        'file': 'MH_Comparative_legaulois_lintransigeant_v3.jsonl',
        'column_mapping': {
            'id_1': 'legaulois_id',
            'id_2': 'lintransigeant_id',
            'doc_1': 'legaulois_doc',
            'doc_2': 'lintransigeant_doc',
            'metadata_1': 'legaulois_metadata',
            'metadata_2': 'lintransigeant_metadata'
        },
        'source1_name': 'Le Gaulois',
        'source2_name': "L'Intransigeant",
        'question_type': 'MH - Comparative'
    },
    {
        'name': 'Generic_LeGaulois_LIntransigeant',
        'file': 'MH_Generic_legaulois_lintransigeant_v3.jsonl',
        'column_mapping': {
            'id_1': 'legaulois_id',
            'id_2': 'lintransigeant_id',
            'doc_1': 'legaulois_doc',
            'doc_2': 'lintransigeant_doc',
            'metadata_1': 'legaulois_metadata',
            'metadata_2': 'lintransigeant_metadata'
        },
        'source1_name': 'Le Gaulois',
        'source2_name': "L'Intransigeant",
        'question_type': 'MH - Generic'
    }
]

# Collection configurations derived from config
COLLECTION_CONFIGS = [
    {'name': c['name'], 'label': c['label']} for c in NEWSPAPER_CONFIGS
] + [{'name': DEBATS_COLLECTION_NAME, 'label': 'Les Débats'}]

EMBEDDING_MODEL = COHERE_EMBEDDING_MODEL
N_RESULTS = 10
K_VALUES = [1, 2, 3, 5, 10]


In [2]:
# ========================================================================
# Step 1: Initialize ChromaDB collections
# ========================================================================
print("Step 1: Initializing collections...")

collections = []
for config in COLLECTION_CONFIGS:
    collection = load_chroma_collection(
        collection_path=EMBEDDINGS_PATH,
        collection_name=config['name'],
        embedding_type='cohere',
        api_key=COHERE_API_KEY,
        embedding_model=EMBEDDING_MODEL,
    )
    collections.append(collection)
    print(f"  Loaded collection: {config['label']}")


# ========================================================================
# Step 2: Load and prepare all question datasets
# ========================================================================
print("\nStep 2: Loading question datasets...")

all_questions = []

for dataset_config in QUESTION_DATASETS:
    print(f"\n  Processing: {dataset_config['name']}")
    jsonl_path = os.path.join(QUESTIONS_DIR, dataset_config['file'])

    if not os.path.exists(jsonl_path):
        print(f"    WARNING: File not found - {jsonl_path}")
        continue

    df_questions = create_questions_dataset(
        jsonl_path=jsonl_path,
        column_mapping=dataset_config['column_mapping'],
        source1_name=dataset_config['source1_name'],
        source2_name=dataset_config['source2_name'],
        question_type=dataset_config['question_type'],
    )
    all_questions.append(df_questions)
    print(f"    Loaded {len(df_questions)} questions")

df = pd.concat(all_questions, ignore_index=True)
print(f"\n  Total questions loaded: {len(df)}")
print(f"  Question types: {df['question_type'].value_counts().to_dict()}")


Step 1: Initializing collections...
  Loaded collection: Le Gaulois
  Loaded collection: L'Intransigeant
  Loaded collection: Les Débats

Step 2: Loading question datasets...

  Processing: Generic_LeGaulois_Debats
Created dataset with 220 questions from 77 original entries
    Loaded 220 questions

  Processing: Generic_LIntransigeant_Debats
Created dataset with 199 questions from 79 original entries
    Loaded 199 questions

  Processing: BridgeEntity_LIntransigeant_Debats
Created dataset with 152 questions from 79 original entries
    Loaded 152 questions

  Processing: Comparative_LeGaulois_LIntransigeant
Created dataset with 192 questions from 47 original entries
    Loaded 192 questions

  Processing: Generic_LeGaulois_LIntransigeant
Created dataset with 122 questions from 48 original entries
    Loaded 122 questions

  Total questions loaded: 885
  Question types: {'MH - Generic': 541, 'MH - Comparative': 192, 'MH - BridgeEntity': 152}


In [3]:
# ========================================================================
# Step 3: Perform retrieval
# ========================================================================
print("Step 3: Performing retrieval...")

retrieval_results_file = os.path.join(OUTPUT_DIR, "MH_retrieval_results_all_collections_combined.jsonl")

retriever = RAGRetriever(collections, n_results=N_RESULTS)

perform_retrieval(df, retriever, output_file=retrieval_results_file)


Step 3: Performing retrieval...
Found 1 existing retrieval results
Retrieving for 884 questions...


  0%|          | 0/884 [00:00<?, ?it/s]

100%|██████████| 884/884 [13:16<00:00,  1.11it/s]

Retrieval complete! Results saved to c:\Users\user\Desktop\workdir\ORDER_EMNLP2026\data\retrieval\MH_retrieval_results_all_collections_combined.jsonl


In [4]:
# Evaluate all questions
retrieval_results_file = os.path.join(OUTPUT_DIR, "MH_retrieval_results_all_collections_combined.jsonl")
df_result = pd.DataFrame(read_jsonl_basic(retrieval_results_file))
result = evaluate_and_print_metrics(df_result, k_values=[1, 2, 3, 5, 10, 30])



EVALUATION RESULTS FOR: DATASET

Computing recall metrics...


100%|██████████| 885/885 [00:00<00:00, 5742.63it/s]



Recall Metrics for 885 questions:
  recall@1: 0.182
  recall@2: 0.297
  recall@3: 0.360
  recall@5: 0.443
  recall@10: 0.546
  recall@30: 0.852


Computing Accuracy: 100%|██████████| 885/885 [00:00<00:00, 8959.22it/s]



Accuracy Metrics for 885 questions:
  accuracy@1: 0.000
  accuracy@2: 0.096
  accuracy@3: 0.155
  accuracy@5: 0.251
  accuracy@10: 0.372
  accuracy@30: 0.759


Computing MRR: 100%|██████████| 885/885 [00:00<00:00, 3613.92it/s]


MRR Metrics for 885 questions:
  mrr@1: 0.182
  mrr@2: 0.240
  mrr@3: 0.261
  mrr@5: 0.280
  mrr@10: 0.294
  mrr@30: 0.314


In [22]:
# Evaluate only newspaper × newspaper questions (source_2 is not Les Débats)
dft = df_result.loc[df_result["source_2"] != "Les Débats"]
result_newspapers = evaluate_and_print_metrics(dft)



EVALUATION RESULTS FOR: DATASET

Computing recall metrics...


100%|██████████| 314/314 [00:00<00:00, 19005.04it/s]



Recall Metrics for 314 questions:
  recall@2: 0.123
  recall@3: 0.142
  recall@5: 0.196
  recall@10: 0.298


Computing Accuracy: 100%|██████████| 314/314 [00:00<00:00, 15716.88it/s]



Accuracy Metrics for 314 questions:
  accuracy@2: 0.010
  accuracy@3: 0.010
  accuracy@5: 0.045
  accuracy@10: 0.118


Computing MRR: 100%|██████████| 314/314 [00:00<00:00, 6827.36it/s]


MRR Metrics for 314 questions:
  mrr@2: 0.097
  mrr@3: 0.104
  mrr@5: 0.116
  mrr@10: 0.129


In [23]:
# Evaluate only newspaper × Les Débats questions
dft = df_result.loc[df_result["source_2"] == "Les Débats"]
result_debats = evaluate_and_print_metrics(dft)



EVALUATION RESULTS FOR: DATASET

Computing recall metrics...


100%|██████████| 496/496 [00:00<00:00, 19077.26it/s]



Recall Metrics for 496 questions:
  recall@2: 0.398
  recall@3: 0.487
  recall@5: 0.589
  recall@10: 0.693


Computing Accuracy: 100%|██████████| 496/496 [00:00<00:00, 11391.14it/s]



Accuracy Metrics for 496 questions:
  accuracy@2: 0.147
  accuracy@3: 0.240
  accuracy@5: 0.381
  accuracy@10: 0.526


Computing MRR: 100%|██████████| 496/496 [00:00<00:00, 6977.54it/s]


MRR Metrics for 496 questions:
  mrr@2: 0.326
  mrr@3: 0.355
  mrr@5: 0.379
  mrr@10: 0.393
